# MONAI Ventricular Short Axis 3-Label Inference on ACDC Dataset

This notebook performs cardiac segmentation inference using the pre-trained MONAI VSA-3L model on ACDC testing data.

In [ ]:
# Install required packages
# import sys
# !{sys.executable} -m pip install monai[all] torch matplotlib SimpleITK numpy

In [ ]:
# Imports
import os
import json
import subprocess
import re
from pathlib import Path

import numpy as np
import torch
import monai
import SimpleITK as sitk
import matplotlib.pyplot as plt
import pandas as pd
from medpy.metric import binary

print(f"MONAI version: {monai.__version__}")
print(f"PyTorch version: {torch.__version__}")

## Configuration

**Quick Start:**
- Set `DATASET_MODE = "testing"` for test data evaluation with ground truth
- Set `DATASET_MODE = "training"` for training data inference (ground truth available)
- Outputs are automatically saved to separate directories: `output_testing/` or `output_training/`

### Dataset Information
- **Training**: 100 patients with ground truth (ED and ES frames)
- **Testing**: 50 patients with ground truth (ED and ES frames)
- All outputs are automatically organized into separate directories

In [ ]:
# ============================================================================
# Dataset selection
# ============================================================================
DATASET_MODE = "testing"  # Options: "training" or "testing"

# Paths
ACDC_BASE = os.environ["ACDC_ROOT"]
DATA_DIR = f"{ACDC_BASE}/{DATASET_MODE}"
MODEL_PATH = os.environ["VSA3L_ROOT"] + "/models/model.pt"
BUNDLE_ROOT = os.environ["VSA3L_ROOT"]

# Results folder at project root
RESULTS_DIR = Path(BUNDLE_ROOT) / "results" / f"vsa3l_acdc_{DATASET_MODE}"
PREDICTIONS_DIR = RESULTS_DIR / "predictions"
VISUALIZATIONS_DIR = RESULTS_DIR / "visualizations"
TMP_DIR = RESULTS_DIR / "temp_input"

# CASC-style debug output root (single debug root as requested)
DEBUG_DIR = Path(BUNDLE_ROOT) / "results" / "debug"
DEBUG_TEXT_DIR = DEBUG_DIR / "text"
DEBUG_FIGURES_DIR = DEBUG_DIR / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
VISUALIZATIONS_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_TEXT_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset mode: {DATASET_MODE.upper()}")
print(f"Data directory: {DATA_DIR}")
print(f"Results directory: {RESULTS_DIR}")
print(f"  predictions/: {PREDICTIONS_DIR}")
print(f"  visualizations/: {VISUALIZATIONS_DIR}")
print(f"  temp_input/: {TMP_DIR}")
print(f"Debug directory: {DEBUG_DIR}")
print(f"  text/: {DEBUG_TEXT_DIR}")
print(f"  figures/: {DEBUG_FIGURES_DIR}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

assert os.path.exists(DATA_DIR), f"Data directory not found: {DATA_DIR}"
assert os.path.exists(MODEL_PATH), f"Model file not found: {MODEL_PATH}"
assert os.path.exists(BUNDLE_ROOT), f"Bundle root not found: {BUNDLE_ROOT}"
print("All paths verified")

# ============================================================================
# CASC-style debug event logger (notebook-native trace)
# ============================================================================
import time
import datetime as dt
import resource

RUN_NAME = f"notebook_vsa3l_{DATASET_MODE}"
WORKFLOW_START = dt.datetime.now().astimezone().isoformat()

DEBUG_EVENTS = []
_ACTIVE_DEBUG_EVENTS = {}

def _rss_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024.0 * 1024.0)

def start_debug_event(process_name: str, label: str = "process_low", attempt: int = 1):
    key = (process_name, len(_ACTIVE_DEBUG_EVENTS))
    _ACTIVE_DEBUG_EVENTS[key] = {
        "name": process_name,
        "label": label,
        "attempt": attempt,
        "submit_ts": dt.datetime.now().astimezone(),
        "start_perf": time.perf_counter(),
        "rss_start_gb": _rss_gb(),
    }
    return key

def end_debug_event(key, status: str = "COMPLETED", cpu_percent=np.nan):
    entry = _ACTIVE_DEBUG_EVENTS.pop(key, None)
    if entry is None:
        return
    end_perf = time.perf_counter()
    end_ts = dt.datetime.now().astimezone()
    runtime_seconds = max(0.0, end_perf - entry["start_perf"])
    peak_rss_gb = max(entry["rss_start_gb"], _rss_gb())
    DEBUG_EVENTS.append({
        "name": entry["name"],
        "status": status,
        "realtime": f"{runtime_seconds:.3f}s",
        "%cpu": cpu_percent,
        "peak_rss": f"{peak_rss_gb:.6f}GB",
        "label": entry["label"],
        "attempt": entry["attempt"],
        "submit": entry["submit_ts"].isoformat(),
    })

print("Debug logger initialized")

In [ ]:
print("\n" + "="*80)
print(f"CURRENT CONFIGURATION: {DATASET_MODE.upper()}")
print("="*80)
print(f"Patients found: {len(list(Path(DATA_DIR).glob('patient*')))}")
print(f"Predictions directory: {PREDICTIONS_DIR}")
print(f"Visualizations directory: {VISUALIZATIONS_DIR}")
print("="*80)

## MONAI Bundle Inference

In [ ]:
# Prepare ACDC dataset file list
patient_folders = sorted([d for d in Path(DATA_DIR).iterdir() if d.is_dir() and d.name.startswith('patient')])
acdc_files = []

for patient_folder in patient_folders:
    image_4d_path = patient_folder / f"{patient_folder.name}_4d.nii.gz"
    if image_4d_path.exists():
        acdc_files.append(str(image_4d_path))

print(f"Found {len(acdc_files)} patient files")

In [ ]:
import nibabel as nib

# Extract 2D slices for ED and ES frames
slice_dir = TMP_DIR / "slices_np"
slice_dir.mkdir(exist_ok=True)

slice_items = []  # patient, frame_type, slice_idx, npy_path, original_4d, voxelspacing_2d
print("Extracting 2D slices for ED and ES frames...")

for patient_file in acdc_files:
    patient_path = Path(patient_file)
    patient_name = patient_path.parent.name
    preprocess_key = start_debug_event(f"VSA3L_PREPROCESS ({patient_name})", label="process_low")
    try:
        info_file = patient_path.parent / "Info.cfg"
        ed_frame = None
        es_frame = None
        if info_file.exists():
            with open(info_file, 'r') as f:
                for line in f:
                    if line.startswith('ED:'):
                        ed_frame = int(line.split(':')[1].strip())
                    elif line.startswith('ES:'):
                        es_frame = int(line.split(':')[1].strip())

        img_4d = nib.load(patient_file)
        data_4d = img_4d.get_fdata()
        voxelspacing_xyz = img_4d.header.get_zooms()[:3]
        voxelspacing_2d = (float(voxelspacing_xyz[1]), float(voxelspacing_xyz[0]))

        if len(data_4d.shape) != 4:
            end_debug_event(preprocess_key, status="FAILED")
            continue

        if ed_frame is None:
            ed_frame = 0
        if es_frame is None:
            es_frame = data_4d.shape[-1] - 1

        for frame_type, frame_idx in {"ED": ed_frame, "ES": es_frame}.items():
            if frame_idx < 0 or frame_idx >= data_4d.shape[-1]:
                continue
            data_3d = data_4d[:, :, :, frame_idx]
            for slice_idx in range(data_3d.shape[2]):
                slice_2d = data_3d[:, :, slice_idx]
                if np.sum(slice_2d > 0) < 100:
                    continue
                slice_2d = np.squeeze(slice_2d)
                if len(slice_2d.shape) != 2:
                    continue

                slice_filename = f"{patient_name}_{frame_type}_slice_{slice_idx:02d}.npy"
                slice_path = slice_dir / slice_filename
                np.save(slice_path, slice_2d.astype(np.float32))

                slice_items.append({
                    'patient': patient_name,
                    'frame_type': frame_type,
                    'slice_idx': int(slice_idx),
                    'npy_path': str(slice_path),
                    'original_4d': str(patient_file),
                    'voxelspacing_2d': voxelspacing_2d,
                })
        end_debug_event(preprocess_key, status="COMPLETED")
    except Exception:
        end_debug_event(preprocess_key, status="FAILED")
        raise

print(f"Extracted {len(slice_items)} slices (ED+ES) from {len(acdc_files)} patients")

## Execute Inference

In [ ]:
os.chdir(BUNDLE_ROOT)

print("Loading model...")
load_model_key = start_debug_event("VSA3L_LOAD_MODEL", label="process_low")
parser = monai.bundle.load_bundle_config(".", "inference.json")
net = parser.get_parsed_content("network_def")
net.load_state_dict(torch.load("models/model.pt", map_location=device, weights_only=False))
net = net.to(device)
net.eval()
end_debug_event(load_model_key, status="COMPLETED")

print("Running inference on extracted ED/ES slices...")
segmentation_files = []

patient_event_keys = {}
for item in slice_items:
    patient_name = item['patient']
    if patient_name not in patient_event_keys:
        patient_event_keys[patient_name] = start_debug_event(
            f"VSA3L_SEGMENT ({patient_name})",
            label="process_gpu"
        )
    slice_path = Path(item['npy_path'])
    im = np.load(slice_path)

    from skimage.transform import resize
    if im.shape != (256, 256):
        im = resize(im, (256, 256), preserve_range=True, anti_aliasing=True)

    denom = float(im.max())
    if denom == 0.0:
        continue
    inputd = torch.from_numpy(im / denom).float().to(device)

    with torch.no_grad():
        input_tensor = inputd[None, None, :, :]
        pred = net(input_tensor)
        pred = torch.softmax(pred[0], dim=0)
        seg = torch.argmax(pred, dim=0).cpu().numpy()

    # Remap labels: model (1=LV, 2=Myo, 3=RV) -> ACDC (1=RV, 2=Myo, 3=LV)
    seg_remapped = np.zeros_like(seg)
    seg_remapped[seg == 1] = 3
    seg_remapped[seg == 2] = 2
    seg_remapped[seg == 3] = 1

    seg_filename = f"{slice_path.stem}_seg.nii.gz"
    seg_path = PREDICTIONS_DIR / seg_filename
    seg_img = sitk.GetImageFromArray(seg_remapped.astype(np.uint8))
    sitk.WriteImage(seg_img, str(seg_path))
    segmentation_files.append(str(seg_path))

for key in patient_event_keys.values():
    end_debug_event(key, status="COMPLETED")

print(f"Generated {len(segmentation_files)} segmentation files")
print(f"Predictions saved to: {PREDICTIONS_DIR}")

## Results Visualization

In [ ]:
import re

def load_nifti_image(filepath):
    """Load NIfTI image using SimpleITK."""
    image = sitk.ReadImage(str(filepath))
    array = sitk.GetArrayFromImage(image)
    return array, image

def _strip_nii_suffix(name: str) -> str:
    """Strip .nii.gz or .nii from filename."""
    if name.endswith(".nii.gz"):
        return name[:-7]
    if name.endswith(".nii"):
        return name[:-4]
    return name

# Quick visualization: show the first available prediction
output_files = sorted(PREDICTIONS_DIR.glob("*_seg.nii.gz"))

if not output_files:
    print(f"No segmentation files found in: {PREDICTIONS_DIR}")
else:
    seg_file = output_files[0]
    # Properly strip extensions: .nii.gz then _seg
    seg_stem = _strip_nii_suffix(seg_file.name)
    seg_stem = re.sub(r"_seg$", "", seg_stem)

    patient_name = "UNK"
    frame_type = "UNK"
    slice_num = 0

    if "_slice_" in seg_stem:
        left, slice_part = seg_stem.split("_slice_", 1)
        parts = left.split("_")
        if len(parts) >= 1:
            patient_name = parts[0]
        if len(parts) >= 2:
            frame_type = parts[1]
        # Extract only digits from slice_part
        m = re.match(r"(\d+)", slice_part)
        if m:
            slice_num = int(m.group(1))
    else:
        patient_name = seg_stem

    original_path = Path(DATA_DIR) / patient_name / f"{patient_name}_4d.nii.gz"
    if not original_path.exists():
        print(f"Original image not found: {original_path}")
    else:
        seg_img, _ = load_nifti_image(seg_file)

        import nibabel as nib
        nib_img = nib.load(original_path)
        data_4d = nib_img.get_fdata()

        frame_idx = None
        info_file = Path(DATA_DIR) / patient_name / "Info.cfg"
        if info_file.exists():
            with open(info_file, 'r') as f:
                for line in f:
                    if frame_type == "ED" and line.startswith('ED:'):
                        frame_idx = int(line.split(':')[1].strip())
                    if frame_type == "ES" and line.startswith('ES:'):
                        frame_idx = int(line.split(':')[1].strip())

        if len(data_4d.shape) == 4:
            if frame_idx is None:
                frame_idx = data_4d.shape[-1] // 2
            data_3d = data_4d[:, :, :, frame_idx]
            original_slice = data_3d[:, :, slice_num]
        else:
            original_slice = data_4d[:, :, slice_num]

        seg_slice = np.squeeze(seg_img)
        from skimage.transform import resize
        if seg_slice.shape != original_slice.shape:
            seg_slice = resize(seg_slice, original_slice.shape, preserve_range=True, anti_aliasing=False, order=0).astype(np.uint8)

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(original_slice, cmap='gray', origin='lower')
        axes[0].set_title(f'Original - {patient_name} {frame_type} (Slice {slice_num})')
        axes[0].axis('off')

        axes[1].imshow(seg_slice, cmap='jet', origin='lower', vmin=0, vmax=3)
        axes[1].set_title('Segmentation')
        axes[1].axis('off')

        axes[2].imshow(original_slice, cmap='gray', origin='lower')
        seg_masked = np.ma.masked_where(seg_slice == 0, seg_slice)
        axes[2].imshow(seg_masked, alpha=0.6, cmap='jet', origin='lower', vmin=0, vmax=3)
        axes[2].set_title('Overlay')
        axes[2].axis('off')

        plt.tight_layout()
        plt.show()

## Summary

Segmentation results are saved in the output directory with the following labels:
- Label 0: Background
- Label 1: Right ventricle  
- Label 2: Myocardium
- Label 3: Left ventricle

In [ ]:
# Visualize a few segmentations from the predictions folder
import re

def _strip_nii_suffix(name: str) -> str:
    # Handles .nii.gz and .nii
    if name.endswith(".nii.gz"):
        return name[:-7]
    if name.endswith(".nii"):
        return name[:-4]
    return name

def process_segmentation(seg_file_path):
    seg_file = Path(seg_file_path)
    # Robustly remove extensions and optional `_seg` suffix
    seg_stem = _strip_nii_suffix(seg_file.name)
    seg_stem = re.sub(r"_seg$", "", seg_stem)

    if "_slice_" not in seg_stem:
        return

    left, slice_part = seg_stem.split("_slice_", 1)
    m = re.match(r"(\d+)", slice_part)
    if not m:
        return
    slice_num = int(m.group(1))

    parts = left.split("_")
    patient_name = parts[0]
    frame_type = parts[1] if len(parts) > 1 else "UNK"

    original_path = Path(DATA_DIR) / patient_name / f"{patient_name}_4d.nii.gz"
    if not original_path.exists():
        return

    original_img = nib.load(str(original_path))
    original_data = original_img.get_fdata()

    if slice_num >= original_data.shape[2]:
        return

    # Note: frame index selection here is only for visualization.
    # The evaluation code should use true ED/ES indices from Info.cfg.
    if frame_type == "ED":
        frame_idx = 0
    elif frame_type == "ES":
        frame_idx = original_data.shape[3] - 1
    else:
        frame_idx = 0

    original_slice = original_data[:, :, slice_num, frame_idx]

    seg_data = nib.load(str(seg_file)).get_fdata()

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.title(f"Original: {patient_name} {frame_type} Slice {slice_num}")
    plt.imshow(original_slice, cmap="gray")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Prediction")
    plt.imshow(original_slice, cmap="gray")
    plt.imshow(seg_data, alpha=0.5, cmap="jet")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

# List segmentations in predictions dir and visualize a few
seg_output_dir = PREDICTIONS_DIR
output_files = sorted(seg_output_dir.glob("*.nii.gz"))
print(f"Found {len(output_files)} segmentations in {seg_output_dir}")

for file_path in output_files[:5]:
    process_segmentation(file_path)

## Ground Truth Comparison and Evaluation

Evaluate segmentation performance using Dice Similarity Coefficient (DSC) scores. Ground truth is available for both training and testing datasets.

### Important Note on Cardiac Anatomy

Cardiac short-axis MRI captures slices from **apex** (tip) to **base** (top) of the heart:

- **Apical slices** (e.g., slices 0-1): Only LV and myocardium visible. RV is anatomically absent.
- **Mid-ventricular slices** (e.g., slices 2-5): Both LV and RV clearly visible with full myocardium.
- **Basal slices** (e.g., slices 6+): Complex geometry near heart valves and atria.

**Expected Behavior:**
- Low DSC scores on apical slices are expected if the model predicts RV where it doesn't exist
- Best performance should be on mid-ventricular slices where all structures are present
- This reflects a model limitation: it may not handle anatomical variation across slice positions

In [ ]:
print("Exploring ACDC dataset structure...")
print(f"Data directory: {DATA_DIR}\n")

sample_patient = sorted([d for d in Path(DATA_DIR).iterdir() if d.is_dir() and d.name.startswith('patient')])[0]
print(f"Sample patient: {sample_patient.name}")

all_files = sorted(sample_patient.glob("*.nii.gz"))
for file in all_files:
    file_size = file.stat().st_size / 1024
    print(f"  {file.name} ({file_size:.1f} KB)")

gt_files = sorted(sample_patient.glob("*_gt.nii.gz"))

if gt_files:
    sample_gt = gt_files[0]
    gt_img = sitk.ReadImage(str(sample_gt))
    gt_array = sitk.GetArrayFromImage(gt_img)
    print(f"\n{sample_gt.name}")
    print(f"  Shape: {gt_array.shape}")
    print(f"  Labels: {np.unique(gt_array)} (0=BG, 1=RV, 2=Myo, 3=LV)")

In [ ]:
import nibabel as nib

# Build ground-truth slice stack for ED and ES frames to match our predictions
gt_slice_dir = TMP_DIR / "ground_truth_slices"
gt_slice_dir.mkdir(exist_ok=True)

gt_slice_items = []  # list of dicts: patient, frame_type, slice_idx, npy_path, voxelspacing_2d
print(f"Extracting ED/ES ground truth slices from {DATASET_MODE} data...")
print(f"Output: {gt_slice_dir}\n")

patient_folders = sorted([d for d in Path(DATA_DIR).iterdir() if d.is_dir() and d.name.startswith('patient')])
for patient_folder in patient_folders:
    patient_name = patient_folder.name
    gt_key = start_debug_event(f"GT_EXTRACT ({patient_name})", label="process_low")
    try:
        info_file = patient_folder / "Info.cfg"
        ed_frame = None
        es_frame = None
        if info_file.exists():
            with open(info_file, 'r') as f:
                for line in f:
                    if line.startswith('ED:'):
                        ed_frame = int(line.split(':')[1].strip())
                    elif line.startswith('ES:'):
                        es_frame = int(line.split(':')[1].strip())

        gt_files = sorted(patient_folder.glob("*_gt.nii.gz"))
        if not gt_files:
            end_debug_event(gt_key, status="COMPLETED")
            continue

        for gt_file in gt_files:
            gt_nib = nib.load(gt_file)
            gt_data = gt_nib.get_fdata()
            if len(gt_data.shape) != 3:
                continue

            # Determine whether this GT file is ED or ES by filename (frameXX)
            m = re.search(r"frame(\d+)", gt_file.name)
            frame_idx = int(m.group(1)) if m else None
            frame_type = None
            if frame_idx is not None and ed_frame is not None and frame_idx == ed_frame:
                frame_type = "ED"
            if frame_idx is not None and es_frame is not None and frame_idx == es_frame:
                frame_type = "ES"
            if frame_type is None:
                continue

            voxelspacing_xyz = gt_nib.header.get_zooms()[:3]
            voxelspacing_2d = (float(voxelspacing_xyz[1]), float(voxelspacing_xyz[0]))

            for slice_idx in range(gt_data.shape[2]):
                gt_slice_2d = gt_data[:, :, slice_idx]
                if np.sum(gt_slice_2d > 0) < 10:
                    continue
                gt_slice_2d = np.squeeze(gt_slice_2d)
                if len(gt_slice_2d.shape) != 2:
                    continue

                gt_slice_filename = f"{patient_name}_{frame_type}_slice_{slice_idx:02d}_gt.npy"
                gt_slice_path = gt_slice_dir / gt_slice_filename
                np.save(gt_slice_path, gt_slice_2d.astype(np.uint8))

                gt_slice_items.append({
                    'patient': patient_name,
                    'frame_type': frame_type,
                    'slice_idx': int(slice_idx),
                    'npy_path': str(gt_slice_path),
                    'voxelspacing_2d': voxelspacing_2d,
                })
        end_debug_event(gt_key, status="COMPLETED")
    except Exception:
        end_debug_event(gt_key, status="FAILED")
        raise

print(f"Extracted {len(gt_slice_items)} ground truth slices (ED+ES)")

In [ ]:
# Collect segmentation files from predictions and match with ground truth
seg_files = list(PREDICTIONS_DIR.glob("*.nii.gz"))
print(f"Found {len(seg_files)} predicted segmentations in: {PREDICTIONS_DIR}")

def _strip_nii_suffix(name: str) -> str:
    if name.endswith(".nii.gz"):
        return name[:-7]
    if name.endswith(".nii"):
        return name[:-4]
    return name

# Build lookup from gt_slice_items
gt_lookup = {}
for g in gt_slice_items:
    key = (g['patient'], g['frame_type'], g['slice_idx'])
    gt_lookup[key] = g['npy_path']

# Match predictions to ground truth
matched_pairs = []
for seg_file in seg_files:
    seg_stem = _strip_nii_suffix(seg_file.name)
    seg_stem = re.sub(r"_seg$", "", seg_stem)
    
    if "_slice_" not in seg_stem:
        continue
    
    left, slice_part = seg_stem.split("_slice_", 1)
    m = re.match(r"(\d+)", slice_part)
    if not m:
        continue
    slice_idx = int(m.group(1))
    
    parts = left.split("_")
    patient_id = parts[0]
    frame_type = parts[1] if len(parts) > 1 else None
    
    if frame_type not in ("ED", "ES"):
        continue
    
    key = (patient_id, frame_type, slice_idx)
    if key in gt_lookup:
        matched_pairs.append({
            'patient_id': patient_id,
            'frame_type': frame_type,
            'slice_idx': slice_idx,
            'pred_path': seg_file,
            'gt_path': gt_lookup[key],
        })

print(f"Matched {len(matched_pairs)} prediction-GT pairs")

# Define comparison function
def calculate_dice(pred, gt, label):
    pred_bin = (pred == label).astype(np.uint8)
    gt_bin = (gt == label).astype(np.uint8)
    intersection = np.sum(pred_bin & gt_bin)
    union = np.sum(pred_bin) + np.sum(gt_bin)
    if union == 0:
        return 1.0 if np.sum(gt_bin) == 0 else 0.0
    return 2.0 * intersection / union

def calculate_hd95(pred, gt, label, voxelspacing=None):
    pred_bin = (pred == label).astype(np.uint8)
    gt_bin = (gt == label).astype(np.uint8)
    if np.sum(pred_bin) == 0 or np.sum(gt_bin) == 0:
        return np.inf
    try:
        return binary.hd95(pred_bin, gt_bin, voxelspacing=voxelspacing)
    except Exception:
        return np.inf

def compare_segmentations(pred_path, gt_path, voxelspacing_2d=None):
    """Compare prediction and ground truth, return metrics dict."""
    pred_img = sitk.ReadImage(str(pred_path))
    pred = sitk.GetArrayFromImage(pred_img).squeeze()
    gt = np.load(gt_path)
    
    # Resize pred to match gt if needed
    from skimage.transform import resize
    if pred.shape != gt.shape:
        pred = resize(pred, gt.shape, preserve_range=True, anti_aliasing=False, order=0).astype(np.uint8)
    
    dice_rv = calculate_dice(pred, gt, 1)
    dice_myo = calculate_dice(pred, gt, 2)
    dice_lv = calculate_dice(pred, gt, 3)
    
    hd95_rv = calculate_hd95(pred, gt, 1, voxelspacing_2d)
    hd95_myo = calculate_hd95(pred, gt, 2, voxelspacing_2d)
    hd95_lv = calculate_hd95(pred, gt, 3, voxelspacing_2d)
    
    return {
        'dice_rv': dice_rv,
        'dice_myo': dice_myo,
        'dice_lv': dice_lv,
        'dice_mean': (dice_rv + dice_myo + dice_lv) / 3,
        'hd95_rv': hd95_rv,
        'hd95_myo': hd95_myo,
        'hd95_lv': hd95_lv,
        'hd95_mean': np.nanmean([x for x in [hd95_rv, hd95_myo, hd95_lv] if np.isfinite(x)]) if any(np.isfinite(x) for x in [hd95_rv, hd95_myo, hd95_lv]) else np.inf,
    }

print("Comparison functions defined")

In [ ]:
print("Computing metrics (ED vs ES)...")
print("=" * 80)

if len(matched_pairs) == 0:
    raise RuntimeError("No matched pairs found. Run inference and GT extraction first.")

all_results = []
for pair in matched_pairs:
    metric_key = start_debug_event(
        f"COMPUTE_METRICS ({pair['patient_id']} {pair['frame_type']} slice{pair['slice_idx']:02d})",
        label="process_low"
    )
    try:
        # voxelspacing lookup from gt_slice_items (patient+frame_type)
        voxelspacing_2d = None
        # quick lookup (linear scan is fine at this scale)
        for g in gt_slice_items:
            if g['patient'] == pair['patient_id'] and g['frame_type'] == pair['frame_type'] and g['slice_idx'] == pair['slice_idx']:
                voxelspacing_2d = g.get('voxelspacing_2d')
                break

        res = compare_segmentations(pair['pred_path'], pair['gt_path'], voxelspacing_2d=voxelspacing_2d)
        all_results.append({
            'patient_id': pair['patient_id'],
            'frame_type': pair['frame_type'],
            'slice_idx': pair['slice_idx'],
            'filename': pair['pred_path'].name.replace('_seg', ''),
            **res,
        })
        end_debug_event(metric_key, status="COMPLETED")
    except Exception:
        end_debug_event(metric_key, status="FAILED")
        raise

results_df = pd.DataFrame(all_results)
print(f"✓ Computed metrics for {len(results_df)} slices")

# Patient-level ED/ES summary
aggregate_key = start_debug_event("AGGREGATE_RESULTS", label="process_low")
patient_results = []
for patient_id in sorted(results_df['patient_id'].unique()):
    patient_data = results_df[results_df['patient_id'] == patient_id]
    ed_data = patient_data[patient_data['frame_type'] == 'ED']
    es_data = patient_data[patient_data['frame_type'] == 'ES']

    row = {'patient_id': patient_id}
    # aggregate across slices within frame type
    if len(ed_data) > 0:
        row['ed_dice_rv'] = ed_data['dice_rv'].mean()
        row['ed_dice_myo'] = ed_data['dice_myo'].mean()
        row['ed_dice_lv'] = ed_data['dice_lv'].mean()
        row['ed_dice_mean'] = ed_data['dice_mean'].mean()
        row['ed_hd95_rv'] = ed_data['hd95_rv'].replace([np.inf, -np.inf], np.nan).mean()
        row['ed_hd95_myo'] = ed_data['hd95_myo'].replace([np.inf, -np.inf], np.nan).mean()
        row['ed_hd95_lv'] = ed_data['hd95_lv'].replace([np.inf, -np.inf], np.nan).mean()
        row['ed_hd95_mean'] = ed_data['hd95_mean'].replace([np.inf, -np.inf], np.nan).mean()
    if len(es_data) > 0:
        row['es_dice_rv'] = es_data['dice_rv'].mean()
        row['es_dice_myo'] = es_data['dice_myo'].mean()
        row['es_dice_lv'] = es_data['dice_lv'].mean()
        row['es_dice_mean'] = es_data['dice_mean'].mean()
        row['es_hd95_rv'] = es_data['hd95_rv'].replace([np.inf, -np.inf], np.nan).mean()
        row['es_hd95_myo'] = es_data['hd95_myo'].replace([np.inf, -np.inf], np.nan).mean()
        row['es_hd95_lv'] = es_data['hd95_lv'].replace([np.inf, -np.inf], np.nan).mean()
        row['es_hd95_mean'] = es_data['hd95_mean'].replace([np.inf, -np.inf], np.nan).mean()

    row['overall_dice_mean'] = patient_data['dice_mean'].mean()
    row['overall_hd95_mean'] = patient_data['hd95_mean'].replace([np.inf, -np.inf], np.nan).mean()
    patient_results.append(row)

patient_df = pd.DataFrame(patient_results)
end_debug_event(aggregate_key, status="COMPLETED")
print(f"Created patient-level summary for {len(patient_df)} patients")

In [ ]:
def visualize_comparison(pred_path, gt_path, original_path, patient_name, frame_type, slice_num):
    """Visualize prediction vs ground truth comparison."""
    pred_img, _ = load_nifti_image(pred_path)
    gt_slice = np.load(gt_path)
    
    import nibabel as nib
    nib_img = nib.load(original_path)
    data_4d = nib_img.get_fdata()
    
    # Resolve ED/ES frame index from Info.cfg
    frame_idx = None
    info_file = Path(DATA_DIR) / patient_name / "Info.cfg"
    if info_file.exists():
        with open(info_file, 'r') as f:
            for line in f:
                if frame_type == "ED" and line.startswith('ED:'):
                    frame_idx = int(line.split(':')[1].strip())
                if frame_type == "ES" and line.startswith('ES:'):
                    frame_idx = int(line.split(':')[1].strip())
    
    if len(data_4d.shape) == 4:
        if frame_idx is None:
            frame_idx = data_4d.shape[-1] // 2
        data_3d = data_4d[:, :, :, frame_idx]
        original_slice = data_3d[:, :, slice_num]
    else:
        original_slice = data_4d[:, :, slice_num]
    
    from skimage.transform import resize
    if pred_img.shape != gt_slice.shape:
        pred_resized = resize(pred_img, gt_slice.shape, preserve_range=True, anti_aliasing=False, order=0)
        pred_slice = pred_resized.astype(np.uint8)
    else:
        pred_slice = pred_img
    
    dsc_rv = calculate_dice(pred_slice, gt_slice, 1)
    dsc_myo = calculate_dice(pred_slice, gt_slice, 2)
    dsc_lv = calculate_dice(pred_slice, gt_slice, 3)
    dsc_mean = (dsc_rv + dsc_myo + dsc_lv) / 3
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes[0, 0].imshow(original_slice, cmap='gray', origin='lower')
    axes[0, 0].set_title(f'Original - {patient_name} {frame_type} (Slice {slice_num})')
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(pred_slice, cmap='jet', origin='lower', vmin=0, vmax=3)
    axes[0, 1].set_title('Our Prediction')
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(gt_slice, cmap='jet', origin='lower', vmin=0, vmax=3)
    axes[0, 2].set_title('Ground Truth')
    axes[0, 2].axis('off')
    
    axes[1, 0].imshow(original_slice, cmap='gray', origin='lower')
    pred_masked = np.ma.masked_where(pred_slice == 0, pred_slice)
    axes[1, 0].imshow(pred_masked, alpha=0.6, cmap='jet', origin='lower', vmin=0, vmax=3)
    axes[1, 0].set_title('Prediction Overlay')
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(original_slice, cmap='gray', origin='lower')
    gt_masked = np.ma.masked_where(gt_slice == 0, gt_slice)
    axes[1, 1].imshow(gt_masked, alpha=0.6, cmap='jet', origin='lower', vmin=0, vmax=3)
    axes[1, 1].set_title('Ground Truth Overlay')
    axes[1, 1].axis('off')
    
    difference = np.abs(pred_slice.astype(int) - gt_slice.astype(int))
    axes[1, 2].imshow(difference, cmap='hot', origin='lower')
    axes[1, 2].set_title(f'Difference Map\nMean DSC: {dsc_mean:.3f}')
    axes[1, 2].axis('off')
    
    plt.suptitle(
        f'DSC Scores - RV: {dsc_rv:.3f} | Myo: {dsc_myo:.3f} | LV: {dsc_lv:.3f}',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    plt.show()

print("Visualizing comparisons\n")

def _pick_middle_pair(pairs):
    if not pairs:
        return None
    pairs_sorted = sorted(pairs, key=lambda x: x['slice_idx'])
    return pairs_sorted[len(pairs_sorted) // 2]

patient_ids = sorted({p['patient_id'] for p in matched_pairs})
preview_patients = patient_ids[:3]
print(f"Previewing first three patients: {preview_patients}")

for patient in preview_patients:
    original_path = Path(DATA_DIR) / patient / f"{patient}_4d.nii.gz"
    if not original_path.exists():
        print(f"Original image not found: {original_path}")
        continue
    
    print(f"\nPatient: {patient}")
    for frame_type in ["ED", "ES"]:
        frame_pairs = [p for p in matched_pairs if p['patient_id'] == patient and p['frame_type'] == frame_type]
        pair = _pick_middle_pair(frame_pairs)
        if pair is None:
            print(f"  No {frame_type} pairs available")
            continue
        slice_num = pair['slice_idx']
        print(f"  {frame_type} Slice {slice_num}")
        try:
            visualize_comparison(pair['pred_path'], pair['gt_path'], original_path, patient, frame_type, slice_num)
        except Exception as e:
            print(f"Error visualizing {patient} {frame_type} slice {slice_num}: {e}")

In [ ]:
"""Create box plots for Dice scores and HD95."""

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Dice scores - ED
dice_ed_data = [patient_df['ed_dice_rv'], patient_df['ed_dice_myo'], patient_df['ed_dice_lv']]
bp1 = axes[0, 0].boxplot(dice_ed_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
colors = ['red', 'green', 'blue']
for patch, color in zip(bp1['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[0, 0].set_title('ED Dice Scores', fontsize=12)
axes[0, 0].set_ylabel('Dice Score')
axes[0, 0].set_ylim([0, 1])
axes[0, 0].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)

# Dice scores - ES
dice_es_data = [patient_df['es_dice_rv'], patient_df['es_dice_myo'], patient_df['es_dice_lv']]
bp2 = axes[0, 1].boxplot(dice_es_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[0, 1].set_title('ES Dice Scores', fontsize=12)
axes[0, 1].set_ylabel('Dice Score')
axes[0, 1].set_ylim([0, 1])
axes[0, 1].axhline(y=0.9, color='gray', linestyle='--', alpha=0.5)

# HD95 - ED (filter out infinite values)
hd95_ed_data = [
    patient_df['ed_hd95_rv'][np.isfinite(patient_df['ed_hd95_rv'])],
    patient_df['ed_hd95_myo'][np.isfinite(patient_df['ed_hd95_myo'])],
    patient_df['ed_hd95_lv'][np.isfinite(patient_df['ed_hd95_lv'])]
]
bp3 = axes[1, 0].boxplot(hd95_ed_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
for patch, color in zip(bp3['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[1, 0].set_title('ED HD95 (mm)', fontsize=12)
axes[1, 0].set_ylabel('HD95 (mm)')

# HD95 - ES
hd95_es_data = [
    patient_df['es_hd95_rv'][np.isfinite(patient_df['es_hd95_rv'])],
    patient_df['es_hd95_myo'][np.isfinite(patient_df['es_hd95_myo'])],
    patient_df['es_hd95_lv'][np.isfinite(patient_df['es_hd95_lv'])]
]
bp4 = axes[1, 1].boxplot(hd95_es_data, labels=['RV', 'MYO', 'LV'], patch_artist=True)
for patch, color in zip(bp4['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.5)
axes[1, 1].set_title('ES HD95 (mm)', fontsize=12)
axes[1, 1].set_ylabel('HD95 (mm)')

plt.suptitle('VSA-3L ACDC Segmentation Metrics Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()

# Save the figure
boxplot_path = RESULTS_DIR / "metrics_boxplots.png"
plt.savefig(boxplot_path, dpi=150, bbox_inches='tight')
print(f"✓ Box plots saved to: {boxplot_path}")

plt.show()

## CASC-Style Debug Export

Generate CASC-format debugging artifacts with the same filenames and report structure in `results/debug/`.

In [ ]:
import math
import json
from typing import Dict

def infer_architecture(model_name: str) -> str:
    if '__' in model_name:
        return model_name.split('__', 1)[0]
    for prefix in ('cinema', 'nnformer', 'vsa3l'):
        if model_name.startswith(prefix):
            return prefix
    return 'unknown'

def parse_duration_to_seconds(value) -> float:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)

    text = str(value).strip()
    if not text:
        return np.nan

    if text.replace('.', '', 1).isdigit():
        return float(text)

    total = 0.0
    number = ''
    unit_map = {
        'ms': 0.001,
        's': 1.0,
        'm': 60.0,
        'h': 3600.0,
        'd': 86400.0,
    }

    idx = 0
    while idx < len(text):
        ch = text[idx]
        if ch.isdigit() or ch == '.':
            number += ch
            idx += 1
            continue

        if ch.isspace():
            idx += 1
            continue

        if not number:
            idx += 1
            continue

        if text[idx:idx + 2] == 'ms':
            unit = 'ms'
            idx += 2
        else:
            unit = ch
            idx += 1

        if unit in unit_map:
            total += float(number) * unit_map[unit]
        number = ''

    if number:
        total += float(number)

    return total if total > 0 else np.nan

def parse_memory_to_gb(value) -> float:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    text = str(value).strip().lower().replace(' ', '')
    if not text:
        return np.nan

    try:
        return float(text)
    except ValueError:
        pass

    units = {
        'kb': 1 / (1024**2),
        'kib': 1 / (1024**2),
        'mb': 1 / 1024,
        'mib': 1 / 1024,
        'gb': 1.0,
        'gib': 1.0,
        'tb': 1024.0,
        'tib': 1024.0,
        'b': 1 / (1024**3),
    }

    for unit, factor in units.items():
        if text.endswith(unit):
            number = text[:-len(unit)]
            try:
                return float(number) * factor
            except ValueError:
                return np.nan

    return np.nan

def safe_mean(series: pd.Series) -> float:
    clean = pd.to_numeric(series, errors='coerce').dropna()
    return float(clean.mean()) if len(clean) else np.nan

def safe_median(series: pd.Series) -> float:
    clean = pd.to_numeric(series, errors='coerce').dropna()
    return float(clean.median()) if len(clean) else np.nan

def safe_p95(series: pd.Series) -> float:
    clean = pd.to_numeric(series, errors='coerce').dropna()
    return float(np.percentile(clean, 95)) if len(clean) else np.nan

def derive_makespan_seconds(trace_df: pd.DataFrame) -> float:
    if trace_df.empty or 'submit' not in trace_df.columns:
        return np.nan

    df = trace_df.copy()
    submit_ts = pd.to_datetime(df['submit'], errors='coerce')
    runtime_s = df['realtime'].map(parse_duration_to_seconds) if 'realtime' in df.columns else pd.Series(np.nan, index=df.index)
    finish_ts = submit_ts + pd.to_timedelta(runtime_s, unit='s')

    if submit_ts.isna().all() or finish_ts.isna().all():
        return np.nan

    start = submit_ts.min()
    end = finish_ts.max()
    if pd.isna(start) or pd.isna(end):
        return np.nan
    makespan = (end - start).total_seconds()
    return float(makespan) if makespan > 0 else np.nan

def compute_execution_metrics(trace_df: pd.DataFrame, workflow_duration: str, workflow_success: str) -> Dict:
    if trace_df.empty:
        return {
            'workflow_success': str(workflow_success).lower() == 'true',
            'workflow_duration_seconds': parse_duration_to_seconds(workflow_duration),
            'total_tasks': 0,
            'completed_tasks': 0,
            'failed_tasks': 0,
            'task_success_rate': np.nan,
            'retry_rate': np.nan,
            'cpu_time_hours_est': np.nan,
            'memory_gb_hours_est': np.nan,
            'makespan_seconds_trace': np.nan,
            'mean_parallelism_est': np.nan,
            'patient_throughput_per_hour': np.nan,
        }

    df = trace_df.copy()
    df['runtime_seconds'] = df['realtime'].map(parse_duration_to_seconds) if 'realtime' in df.columns else np.nan
    df['cpus_num'] = pd.to_numeric(df['cpus'], errors='coerce') if 'cpus' in df.columns else 1.0
    df['peak_rss_gb'] = df['peak_rss'].map(parse_memory_to_gb) if 'peak_rss' in df.columns else np.nan

    total_tasks = len(df)
    completed_tasks = int((df['status'].astype(str).str.upper() == 'COMPLETED').sum()) if 'status' in df.columns else 0
    failed_tasks = int((df['status'].astype(str).str.upper() == 'FAILED').sum()) if 'status' in df.columns else 0

    attempts = pd.to_numeric(df['attempt'], errors='coerce') if 'attempt' in df.columns else pd.Series(dtype=float)
    retry_count = int((attempts > 1).sum()) if len(attempts) else 0

    workflow_duration_seconds = parse_duration_to_seconds(workflow_duration)
    if np.isnan(workflow_duration_seconds):
        workflow_duration_seconds = derive_makespan_seconds(df)

    runtime_seconds_sum = pd.to_numeric(df['runtime_seconds'], errors='coerce').fillna(0).sum()
    cpu_time_hours = float((df['runtime_seconds'].fillna(0) * df['cpus_num'].fillna(1)).sum() / 3600.0)
    memory_gb_hours = float((df['runtime_seconds'].fillna(0) / 3600.0 * df['peak_rss_gb'].fillna(0)).sum())

    mean_parallelism = np.nan
    if workflow_duration_seconds and workflow_duration_seconds > 0:
        mean_parallelism = float(runtime_seconds_sum / workflow_duration_seconds)

    derived_success = (failed_tasks == 0) if total_tasks > 0 else (str(workflow_success).lower() == 'true')

    return {
        'workflow_success': derived_success,
        'workflow_duration_seconds': workflow_duration_seconds,
        'total_tasks': int(total_tasks),
        'completed_tasks': int(completed_tasks),
        'failed_tasks': int(failed_tasks),
        'task_success_rate': float(completed_tasks / total_tasks) if total_tasks else np.nan,
        'retry_rate': float(retry_count / total_tasks) if total_tasks else np.nan,
        'cpu_time_hours_est': cpu_time_hours,
        'memory_gb_hours_est': memory_gb_hours,
        'makespan_seconds_trace': workflow_duration_seconds,
        'mean_parallelism_est': mean_parallelism,
        'patient_throughput_per_hour': np.nan,
    }

def build_task_profile(trace_df: pd.DataFrame) -> pd.DataFrame:
    if trace_df.empty:
        return pd.DataFrame()

    df = trace_df.copy()
    df['process_name'] = df['name'] if 'name' in df.columns else 'unknown'
    df['runtime_seconds'] = df['realtime'].map(parse_duration_to_seconds) if 'realtime' in df.columns else np.nan
    df['cpu_percent'] = pd.to_numeric(df['%cpu'].astype(str).str.replace('%', '', regex=False), errors='coerce') if '%cpu' in df.columns else np.nan
    df['peak_rss_gb'] = df['peak_rss'].map(parse_memory_to_gb) if 'peak_rss' in df.columns else np.nan

    profile_rows = []
    for process_name, process_df in df.groupby('process_name'):
        statuses = process_df['status'].astype(str).str.upper() if 'status' in process_df.columns else pd.Series(dtype=str)
        completed = int((statuses == 'COMPLETED').sum()) if len(statuses) else 0
        total = len(process_df)

        profile_rows.append({
            'process': process_name,
            'tasks': int(total),
            'success_rate': float(completed / total) if total else np.nan,
            'runtime_mean_s': safe_mean(process_df['runtime_seconds']),
            'runtime_median_s': safe_median(process_df['runtime_seconds']),
            'runtime_p95_s': safe_p95(process_df['runtime_seconds']),
            'cpu_percent_mean': safe_mean(process_df['cpu_percent']),
            'peak_rss_gb_mean': safe_mean(process_df['peak_rss_gb']),
            'peak_rss_gb_max': float(pd.to_numeric(process_df['peak_rss_gb'], errors='coerce').max()) if len(process_df) else np.nan,
            'runtime_hours_total': float(pd.to_numeric(process_df['runtime_seconds'], errors='coerce').fillna(0).sum() / 3600.0),
        })

    profile = pd.DataFrame(profile_rows)
    if not profile.empty:
        profile = profile.sort_values(by='runtime_hours_total', ascending=False)
    return profile

def build_gpu_profile(trace_df: pd.DataFrame, workflow_duration_seconds: float) -> Dict:
    if trace_df.empty:
        return {
            'gpu_task_count': 0,
            'gpu_task_fraction': np.nan,
            'gpu_walltime_hours_est': 0.0,
            'avg_gpu_concurrency_est': np.nan,
            'gpu_process_breakdown': [],
        }

    df = trace_df.copy()
    df['runtime_seconds'] = df['realtime'].map(parse_duration_to_seconds) if 'realtime' in df.columns else np.nan
    label_series = df['label'].astype(str).str.lower() if 'label' in df.columns else pd.Series([''] * len(df))
    name_series = df['name'].astype(str).str.lower() if 'name' in df.columns else pd.Series([''] * len(df))

    gpu_mask = label_series.str.contains('process_gpu', na=False) | name_series.str.contains('segment', na=False)
    gpu_df = df[gpu_mask].copy()

    gpu_task_count = len(gpu_df)
    total_tasks = len(df)
    gpu_walltime_hours = float(pd.to_numeric(gpu_df['runtime_seconds'], errors='coerce').fillna(0).sum() / 3600.0)

    avg_concurrency = np.nan
    if workflow_duration_seconds and workflow_duration_seconds > 0:
        avg_concurrency = float((gpu_walltime_hours * 3600.0) / workflow_duration_seconds)

    breakdown = []
    if gpu_task_count:
        for process_name, process_df in gpu_df.groupby('name' if 'name' in gpu_df.columns else 'status'):
            breakdown.append({
                'process': str(process_name),
                'gpu_tasks': int(len(process_df)),
                'gpu_walltime_hours': float(pd.to_numeric(process_df['runtime_seconds'], errors='coerce').fillna(0).sum() / 3600.0),
                'runtime_median_s': safe_median(process_df['runtime_seconds']),
            })

    return {
        'gpu_task_count': int(gpu_task_count),
        'gpu_task_fraction': float(gpu_task_count / total_tasks) if total_tasks else np.nan,
        'gpu_walltime_hours_est': gpu_walltime_hours,
        'avg_gpu_concurrency_est': avg_concurrency,
        'gpu_process_breakdown': sorted(breakdown, key=lambda x: x['gpu_walltime_hours'], reverse=True),
    }

def compute_scientific_metrics(metrics_df: pd.DataFrame, n_patients_input: int) -> Dict:
    summary = {
        'n_patients_input': n_patients_input,
        'n_patients_with_metrics': 0,
        'coverage_rate': np.nan,
        'n_models_evaluated': 0,
        'best_model_overall_dice': None,
        'best_model_dice_value': np.nan,
        'cross_model_dice_spread': np.nan,
        'architecture_consistency_cv_mean': np.nan,
        'rv_dice_mean': np.nan,
        'myo_dice_mean': np.nan,
        'lv_dice_mean': np.nan,
    }

    if metrics_df.empty:
        return summary

    df = metrics_df.copy()
    if 'architecture' not in df.columns and 'model' in df.columns:
        df['architecture'] = df['model'].astype(str).map(infer_architecture)

    if 'patient_id' in df.columns:
        summary['n_patients_with_metrics'] = int(df['patient_id'].nunique())

    if summary['n_patients_input'] and summary['n_patients_input'] > 0:
        summary['coverage_rate'] = float(summary['n_patients_with_metrics'] / summary['n_patients_input'])

    if 'model' in df.columns:
        summary['n_models_evaluated'] = int(df['model'].nunique())

    if 'overall_dice_mean' in df.columns and 'model' in df.columns:
        model_perf = df.groupby('model', as_index=False)['overall_dice_mean'].mean().sort_values('overall_dice_mean', ascending=False)
        if not model_perf.empty:
            best_row = model_perf.iloc[0]
            summary['best_model_overall_dice'] = str(best_row['model'])
            summary['best_model_dice_value'] = float(best_row['overall_dice_mean'])
            if len(model_perf) > 1:
                summary['cross_model_dice_spread'] = float(model_perf['overall_dice_mean'].max() - model_perf['overall_dice_mean'].min())

    if 'overall_dice_mean' in df.columns and 'architecture' in df.columns:
        cvs = []
        for _, sub in df.groupby('architecture'):
            values = pd.to_numeric(sub['overall_dice_mean'], errors='coerce').dropna()
            if len(values) >= 2 and values.mean() > 0:
                cvs.append(float(values.std() / values.mean()))
        if cvs:
            summary['architecture_consistency_cv_mean'] = float(np.mean(cvs))

    for structure in ('rv', 'myo', 'lv'):
        col_candidates = [f'ed_dice_{structure}', f'es_dice_{structure}', f'dice_{structure}']
        vals = []
        for col in col_candidates:
            if col in df.columns:
                vals.extend(pd.to_numeric(df[col], errors='coerce').dropna().tolist())
        if vals:
            summary[f'{structure}_dice_mean'] = float(np.mean(vals))

    return summary

def write_markdown_report(output_path: Path, run_name: str, model_selection: str, workflow_start: str, execution: Dict, gpu_profile: Dict, scientific: Dict):
    lines = []
    lines.append('# CASC Debug Report')
    lines.append('')
    lines.append('## Run Context')
    lines.append(f'- Run name: {run_name}')
    lines.append(f'- Models requested: {model_selection}')
    lines.append(f'- Workflow start: {workflow_start}')
    lines.append(f'- Workflow success: {execution.get("workflow_success")}')
    lines.append('')

    lines.append('## Execution & Reliability')
    lines.append(f'- Workflow duration (s): {execution.get("workflow_duration_seconds")}')
    lines.append(f'- Total tasks: {execution.get("total_tasks")}')
    lines.append(f'- Task success rate: {execution.get("task_success_rate"):.4f}' if not math.isnan(execution.get('task_success_rate', np.nan)) else '- Task success rate: NA')
    lines.append(f'- Retry rate: {execution.get("retry_rate"):.4f}' if not math.isnan(execution.get('retry_rate', np.nan)) else '- Retry rate: NA')
    lines.append(f'- Estimated CPU time (core-hours): {execution.get("cpu_time_hours_est"):.3f}' if not math.isnan(execution.get('cpu_time_hours_est', np.nan)) else '- Estimated CPU time (core-hours): NA')
    lines.append(f'- Estimated memory pressure (GB-hours from peak RSS): {execution.get("memory_gb_hours_est"):.3f}' if not math.isnan(execution.get('memory_gb_hours_est', np.nan)) else '- Estimated memory pressure (GB-hours from peak RSS): NA')
    lines.append(f'- Mean parallelism estimate: {execution.get("mean_parallelism_est"):.3f}' if not math.isnan(execution.get('mean_parallelism_est', np.nan)) else '- Mean parallelism estimate: NA')
    throughput = execution.get('patient_throughput_per_hour', np.nan)
    lines.append(f'- Patient throughput (patients/hour): {throughput:.3f}' if not math.isnan(throughput) else '- Patient throughput (patients/hour): NA')
    lines.append('')

    lines.append('## GPU Scalability')
    lines.append(f'- GPU task count: {gpu_profile.get("gpu_task_count")}')
    gpu_fraction = gpu_profile.get('gpu_task_fraction', np.nan)
    lines.append(f'- GPU task fraction: {gpu_fraction:.4f}' if not math.isnan(gpu_fraction) else '- GPU task fraction: NA')
    lines.append(f'- Estimated GPU walltime (hours): {gpu_profile.get("gpu_walltime_hours_est"):.3f}')
    avg_gpu = gpu_profile.get('avg_gpu_concurrency_est', np.nan)
    lines.append(f'- Average GPU concurrency estimate: {avg_gpu:.3f}' if not math.isnan(avg_gpu) else '- Average GPU concurrency estimate: NA')
    lines.append('')

    lines.append('## Scientific Utility')
    lines.append(f'- Input patient count: {scientific.get("n_patients_input")}')
    lines.append(f'- Evaluated patient count: {scientific.get("n_patients_with_metrics")}')
    coverage = scientific.get('coverage_rate', np.nan)
    lines.append(f'- Coverage rate: {coverage:.4f}' if not math.isnan(coverage) else '- Coverage rate: NA')
    lines.append(f'- Models evaluated: {scientific.get("n_models_evaluated")}')
    lines.append(f'- Best model by mean overall Dice: {scientific.get("best_model_overall_dice")} ({scientific.get("best_model_dice_value"):.4f})' if not math.isnan(scientific.get('best_model_dice_value', np.nan)) else '- Best model by mean overall Dice: NA')
    spread = scientific.get('cross_model_dice_spread', np.nan)
    lines.append(f'- Cross-model Dice spread: {spread:.4f}' if not math.isnan(spread) else '- Cross-model Dice spread: NA')
    consistency = scientific.get('architecture_consistency_cv_mean', np.nan)
    lines.append(f'- Architecture consistency CV (lower is better): {consistency:.4f}' if not math.isnan(consistency) else '- Architecture consistency CV (lower is better): NA')
    for structure in ('rv', 'myo', 'lv'):
        val = scientific.get(f'{structure}_dice_mean', np.nan)
        lines.append(f'- Mean {structure.upper()} Dice: {val:.4f}' if not math.isnan(val) else f'- Mean {structure.upper()} Dice: NA')

    output_path.write_text('\n'.join(lines) + '\n')

def make_figures(task_profile: pd.DataFrame, gpu_profile: Dict, metrics_df: pd.DataFrame, figures_dir: Path):
    figures_dir.mkdir(parents=True, exist_ok=True)

    if not task_profile.empty:
        top = task_profile.head(12)
        plt.figure(figsize=(12, 6))
        plt.bar(top['process'], top['runtime_hours_total'], color='#4e79a7')
        plt.title('Top Process Runtime Contribution')
        plt.ylabel('Runtime (hours)')
        plt.xlabel('Process')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(figures_dir / 'runtime_by_process.png', dpi=160)
        plt.close()

    breakdown = gpu_profile.get('gpu_process_breakdown', [])
    if breakdown:
        gpu_df = pd.DataFrame(breakdown)
        plt.figure(figsize=(10, 5))
        plt.bar(gpu_df['process'], gpu_df['gpu_walltime_hours'], color='#f28e2b')
        plt.title('Estimated GPU Walltime by Process')
        plt.ylabel('GPU walltime (hours)')
        plt.xlabel('Process')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(figures_dir / 'gpu_walltime_by_process.png', dpi=160)
        plt.close()

    if not metrics_df.empty and 'overall_dice_mean' in metrics_df.columns and 'model' in metrics_df.columns:
        model_perf = metrics_df.groupby('model', as_index=False)['overall_dice_mean'].mean().sort_values('overall_dice_mean', ascending=False)
        plt.figure(figsize=(10, 5))
        plt.bar(model_perf['model'], model_perf['overall_dice_mean'], color='#59a14f')
        plt.title('Mean Overall Dice by Model')
        plt.ylabel('Overall Dice')
        plt.xlabel('Model')
        plt.ylim(0, 1)
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.savefig(figures_dir / 'overall_dice_by_model.png', dpi=160)
        plt.close()

# Build CASC-like trace dataframe from notebook events
trace_df = pd.DataFrame(DEBUG_EVENTS) if len(DEBUG_EVENTS) else pd.DataFrame(columns=['name', 'status', 'realtime', '%cpu', 'peak_rss', 'label', 'attempt', 'submit'])
if not trace_df.empty:
    trace_df['cpus'] = 1

# Build metrics snapshot in CASC shape
metrics_snapshot = patient_df.copy() if 'patient_df' in globals() else pd.DataFrame()
if not metrics_snapshot.empty:
    metrics_snapshot['model'] = 'vsa3l__model'
    metrics_snapshot['architecture'] = 'vsa3l'

# Compute summary blocks
workflow_duration_seconds = derive_makespan_seconds(trace_df)
workflow_duration_str = f"{workflow_duration_seconds:.3f}s" if np.isfinite(workflow_duration_seconds) else ""
execution = compute_execution_metrics(trace_df, workflow_duration_str, str(True))
gpu_profile = build_gpu_profile(trace_df, execution.get('workflow_duration_seconds', np.nan))
n_patients_input = len(list(Path(DATA_DIR).glob('patient*')))
scientific = compute_scientific_metrics(metrics_snapshot, n_patients_input=n_patients_input)

if n_patients_input and execution.get('workflow_duration_seconds') and execution['workflow_duration_seconds'] > 0:
    execution['patient_throughput_per_hour'] = float((n_patients_input * 3600.0) / execution['workflow_duration_seconds'])

summary = {
    'run_name': RUN_NAME,
    'workflow_start': WORKFLOW_START,
    'models_requested': 'VSA3L',
    'execution': execution,
    'gpu': {k: v for k, v in gpu_profile.items() if k != 'gpu_process_breakdown'},
    'scientific': scientific,
}

# Persist CASC-style artifacts
task_profile = build_task_profile(trace_df)
gpu_breakdown = pd.DataFrame(gpu_profile.get('gpu_process_breakdown', []))

with open(DEBUG_TEXT_DIR / 'debug_metrics_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)

if not task_profile.empty:
    task_profile.to_csv(DEBUG_TEXT_DIR / 'task_profile.csv', index=False)
if not gpu_breakdown.empty:
    gpu_breakdown.to_csv(DEBUG_TEXT_DIR / 'gpu_profile.csv', index=False)
if not metrics_snapshot.empty:
    preferred_cols = [
        'patient_id', 'model', 'architecture',
        'ed_dice_rv', 'ed_hd95_rv', 'ed_dice_myo', 'ed_hd95_myo', 'ed_dice_lv', 'ed_hd95_lv',
        'es_dice_rv', 'es_hd95_rv', 'es_dice_myo', 'es_hd95_myo', 'es_dice_lv', 'es_hd95_lv',
        'overall_dice_mean'
    ]
    present_cols = [c for c in preferred_cols if c in metrics_snapshot.columns]
    metrics_snapshot[present_cols].to_csv(DEBUG_TEXT_DIR / 'metrics_snapshot.csv', index=False)

write_markdown_report(
    output_path=DEBUG_TEXT_DIR / 'debug_report.md',
    run_name=RUN_NAME,
    model_selection='VSA3L',
    workflow_start=WORKFLOW_START,
    execution=execution,
    gpu_profile=gpu_profile,
    scientific=scientific,
)

make_figures(
    task_profile=task_profile,
    gpu_profile=gpu_profile,
    metrics_df=metrics_snapshot,
    figures_dir=DEBUG_FIGURES_DIR,
)

print(f"Debug summary written to: {DEBUG_TEXT_DIR / 'debug_metrics_summary.json'}")
print(f"Debug markdown report written to: {DEBUG_TEXT_DIR / 'debug_report.md'}")
print(f"Debug text directory: {DEBUG_TEXT_DIR}")
print(f"Debug figures directory: {DEBUG_FIGURES_DIR}")

In [ ]:
import math
import json
from typing import Dict

def infer_architecture(model_name: str) -> str:
    if '__' in model_name:
        return model_name.split('__', 1)[0]
    for prefix in ('cinema', 'nnformer', 'vsa3l'):
        if model_name.startswith(prefix):
            return prefix
    return 'unknown'

def parse_duration_to_seconds(value) -> float:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    if isinstance(value, (int, float)):
        return float(value)

    text = str(value).strip()
    if not text:
        return np.nan

    if text.replace('.', '', 1).isdigit():
        return float(text)

    total = 0.0
    number = ''
    unit_map = {
        'ms': 0.001,
        's': 1.0,
        'm': 60.0,
        'h': 3600.0,
        'd': 86400.0,
    }

    idx = 0
    while idx < len(text):
        ch = text[idx]
        if ch.isdigit() or ch == '.':
            number += ch
            idx += 1
            continue

        if ch.isspace():
            idx += 1
            continue

        if not number:
            idx += 1
            continue

        if text[idx:idx + 2] == 'ms':
            unit = 'ms'
            idx += 2
        else:
            unit = ch
            idx += 1

        if unit in unit_map:
            total += float(number) * unit_map[unit]
        number = ''

    if number:
        total += float(number)

    return total if total > 0 else np.nan

def parse_memory_to_gb(value) -> float:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    text = str(value).strip().lower().replace(' ', '')
    if not text:
        return np.nan

    try:
        return float(text)
    except ValueError:
        pass

    units = {
        'kb': 1 / (1024**2),
        'kib': 1 / (1024**2),
        'mb': 1 / 1024,
        'mib': 1 / 1024,
        'gb': 1.0,
        'gib': 1.0,
        'tb': 1024.0,
        'tib': 1024.0,
        'b': 1 / (1024**3),
    }

    for unit, factor in units.items():
        if text.endswith(unit):
            number = text[:-len(unit)]
            try:
                return float(number) * factor
            except ValueError:
                return np.nan

    return np.nan

def safe_mean(series: pd.Series) -> float:
    clean = pd.to_numeric(series, errors='coerce').dropna()
    return float(clean.mean()) if len(clean) else np.nan

def safe_median(series: pd.Series) -> float:
    clean = pd.to_numeric(series, errors='coerce').dropna()
    return float(clean.median()) if len(clean) else np.nan

def safe_p95(series: pd.Series) -> float:
    clean = pd.to_numeric(series, errors='coerce').dropna()
    return float(np.percentile(clean, 95)) if len(clean) else np.nan

def derive_makespan_seconds(trace_df: pd.DataFrame) -> float:
    if trace_df.empty or 'submit' not in trace_df.columns:
        return np.nan

    df = trace_df.copy()
    submit_ts = pd.to_datetime(df['submit'], errors='coerce')
    runtime_s = df['realtime'].map(parse_duration_to_seconds) if 'realtime' in df.columns else pd.Series(np.nan, index=df.index)
    finish_ts = submit_ts + pd.to_timedelta(runtime_s, unit='s')

    if submit_ts.isna().all() or finish_ts.isna().all():
        return np.nan

    start = submit_ts.min()
    end = finish_ts.max()
    if pd.isna(start) or pd.isna(end):
        return np.nan
    makespan = (end - start).total_seconds()
    return float(makespan) if makespan > 0 else np.nan

def compute_execution_metrics(trace_df: pd.DataFrame, workflow_duration: str, workflow_success: str) -> Dict:
    if trace_df.empty:
        return {
            'workflow_success': str(workflow_success).lower() == 'true',
            'workflow_duration_seconds': parse_duration_to_seconds(workflow_duration),
            'total_tasks': 0,
            'completed_tasks': 0,
            'failed_tasks': 0,
            'task_success_rate': np.nan,
            'retry_rate': np.nan,
            'cpu_time_hours_est': np.nan,
            'memory_gb_hours_est': np.nan,
            'makespan_seconds_trace': np.nan,
            'mean_parallelism_est': np.nan,
            'patient_throughput_per_hour': np.nan,
        }

    df = trace_df.copy()
    df['runtime_seconds'] = df['realtime'].map(parse_duration_to_seconds) if 'realtime' in df.columns else np.nan
    df['cpus_num'] = pd.to_numeric(df['cpus'], errors='coerce') if 'cpus' in df.columns else 1.0
    df['peak_rss_gb'] = df['peak_rss'].map(parse_memory_to_gb) if 'peak_rss' in df.columns else np.nan

    total_tasks = len(df)
    completed_tasks = int((df['status'].astype(str).str.upper() == 'COMPLETED').sum()) if 'status' in df.columns else 0
    failed_tasks = int((df['status'].astype(str).str.upper() == 'FAILED').sum()) if 'status' in df.columns else 0

    attempts = pd.to_numeric(df['attempt'], errors='coerce') if 'attempt' in df.columns else pd.Series(dtype=float)
    retry_count = int((attempts > 1).sum()) if len(attempts) else 0

    workflow_duration_seconds = parse_duration_to_seconds(workflow_duration)
    if np.isnan(workflow_duration_seconds):
        workflow_duration_seconds = derive_makespan_seconds(df)

    runtime_seconds_sum = pd.to_numeric(df['runtime_seconds'], errors='coerce').fillna(0).sum()
    cpu_time_hours = float((df['runtime_seconds'].fillna(0) * df['cpus_num'].fillna(1)).sum() / 3600.0)
    memory_gb_hours = float((df['runtime_seconds'].fillna(0) / 3600.0 * df['peak_rss_gb'].fillna(0)).sum())

    mean_parallelism = np.nan
    if workflow_duration_seconds and workflow_duration_seconds > 0:
        mean_parallelism = float(runtime_seconds_sum / workflow_duration_seconds)

    derived_success = (failed_tasks == 0) if total_tasks > 0 else (str(workflow_success).lower() == 'true')

    return {
        'workflow_success': derived_success,
        'workflow_duration_seconds': workflow_duration_seconds,
        'total_tasks': int(total_tasks),
        'completed_tasks': int(completed_tasks),
        'failed_tasks': int(failed_tasks),
        'task_success_rate': float(completed_tasks / total_tasks) if total_tasks else np.nan,
        'retry_rate': float(retry_count / total_tasks) if total_tasks else np.nan,
        'cpu_time_hours_est': cpu_time_hours,
        'memory_gb_hours_est': memory_gb_hours,
        'makespan_seconds_trace': workflow_duration_seconds,
        'mean_parallelism_est': mean_parallelism,
        'patient_throughput_per_hour': np.nan,
    }

def build_task_profile(trace_df: pd.DataFrame) -> pd.DataFrame:
    if trace_df.empty:
        return pd.DataFrame()

    df = trace_df.copy()
    df['process_name'] = df['name'] if 'name' in df.columns else 'unknown'
    df['runtime_seconds'] = df['realtime'].map(parse_duration_to_seconds) if 'realtime' in df.columns else np.nan
    df['cpu_percent'] = pd.to_numeric(df['%cpu'].astype(str).str.replace('%', '', regex=False), errors='coerce') if '%cpu' in df.columns else np.nan
    df['peak_rss_gb'] = df['peak_rss'].map(parse_memory_to_gb) if 'peak_rss' in df.columns else np.nan

    profile_rows = []
    for process_name, process_df in df.groupby('process_name'):
        statuses = process_df['status'].astype(str).str.upper() if 'status' in df.columns else pd.Series(dtype=str)
        completed = int((statuses == 'COMPLETED').sum()) if len(statuses) else 0
        total = len(process_df)

        profile_rows.append({
            'process': process_name,
            'tasks': int(total),
            'success_rate': float(completed / total) if total else np.nan,
            'runtime_mean_s': safe_mean(process_df['runtime_seconds']),
            'runtime_median_s': safe_median(process_df['runtime_seconds']),
            'runtime_p95_s': safe_p95(process_df['runtime_seconds']),
            'cpu_percent_mean': safe_mean(process_df['cpu_percent']),
            'peak_rss_gb_mean': safe_mean(process_df['peak_rss_gb']),
            'peak_rss_gb_max': float(pd.to_numeric(process_df['peak_rss_gb'], errors='coerce').max()) if len(process_df) else np.nan,
            'runtime_hours_total': float(pd.to_numeric(process_df['runtime_seconds'], errors='coerce').fillna(0).sum() / 3600.0),
        })

    profile = pd.DataFrame(profile_rows)
    if not profile.empty:
        profile = profile.sort_values(by='runtime_hours_total', ascending=False)
    return profile

def build_gpu_profile(trace_df: pd.DataFrame, workflow_duration_seconds: float) -> Dict:
    if trace_df.empty:
        return {
            'gpu_task_count': 0,
            'gpu_task_fraction': np.nan,
            'gpu_walltime_hours_est': 0.0,
            'avg_gpu_concurrency_est': np.nan,
            'gpu_process_breakdown': [],
        }

    df = trace_df.copy()
    df['runtime_seconds'] = df['realtime'].map(parse_duration_to_seconds) if 'realtime' in df.columns else np.nan
    label_series = df['label'].astype(str).str.lower() if 'label' in df.columns else pd.Series([''] * len(df))
    name_series = df['name'].astype(str).str.lower() if 'name' in df.columns else pd.Series([''] * len(df))

    gpu_mask = label_series.str.contains('process_gpu', na=False) | name_series.str.contains('segment', na=False)
    gpu_df = df[gpu_mask].copy()

    gpu_task_count = len(gpu_df)
    total_tasks = len(df)
    gpu_walltime_hours = float(pd.to_numeric(gpu_df['runtime_seconds'], errors='coerce').fillna(0).sum() / 3600.0)

    avg_concurrency = np.nan
    if workflow_duration_seconds and workflow_duration_seconds > 0:
        avg_concurrency = float((gpu_walltime_hours * 3600.0) / workflow_duration_seconds)

    breakdown = []
    if gpu_task_count:
        for process_name, process_df in gpu_df.groupby('name' if 'name' in gpu_df.columns else 'status'):
            breakdown.append({
                'process': str(process_name),
                'gpu_tasks': int(len(process_df)),
                'gpu_walltime_hours': float(pd.to_numeric(process_df['runtime_seconds'], errors='coerce').fillna(0).sum() / 3600.0),
                'runtime_median_s': safe_median(process_df['runtime_seconds']),
            })

    return {
        'gpu_task_count': int(gpu_task_count),
        'gpu_task_fraction': float(gpu_task_count / total_tasks) if total_tasks else np.nan,
        'gpu_walltime_hours_est': gpu_walltime_hours,
        'avg_gpu_concurrency_est': avg_concurrency,
        'gpu_process_breakdown': sorted(breakdown, key=lambda x: x['gpu_walltime_hours'], reverse=True),
    }

def compute_scientific_metrics(metrics_df: pd.DataFrame, n_patients_input: int) -> Dict:
    summary = {
        'n_patients_input': n_patients_input,
        'n_patients_with_metrics': 0,
        'coverage_rate': np.nan,
        'n_models_evaluated': 0,
        'best_model_overall_dice': None,
        'best_model_dice_value': np.nan,
        'cross_model_dice_spread': np.nan,
        'architecture_consistency_cv_mean': np.nan,
        'rv_dice_mean': np.nan,
        'myo_dice_mean': np.nan,
        'lv_dice_mean': np.nan,
    }

    if metrics_df.empty:
        return summary

    df = metrics_df.copy()
    if 'architecture' not in df.columns and 'model' in df.columns:
        df['architecture'] = df['model'].astype(str).map(infer_architecture)

    if 'patient_id' in df.columns:
        summary['n_patients_with_metrics'] = int(df['patient_id'].nunique())

    if summary['n_patients_input'] and summary['n_patients_input'] > 0:
        summary['coverage_rate'] = float(summary['n_patients_with_metrics'] / summary['n_patients_input'])

    if 'model' in df.columns:
        summary['n_models_evaluated'] = int(df['model'].nunique())

    if 'overall_dice_mean' in df.columns and 'model' in df.columns:
        model_perf = df.groupby('model', as_index=False)['overall_dice_mean'].mean().sort_values('overall_dice_mean', ascending=False)
        if not model_perf.empty:
            best_row = model_perf.iloc[0]
            summary['best_model_overall_dice'] = str(best_row['model'])
            summary['best_model_dice_value'] = float(best_row['overall_dice_mean'])
            if len(model_perf) > 1:
                summary['cross_model_dice_spread'] = float(model_perf['overall_dice_mean'].max() - model_perf['overall_dice_mean'].min())

    if 'overall_dice_mean' in df.columns and 'architecture' in df.columns:
        cvs = []
        for _, sub in df.groupby('architecture'):
            values = pd.to_numeric(sub['overall_dice_mean'], errors='coerce').dropna()
            if len(values) >= 2 and values.mean() > 0:
                cvs.append(float(values.std() / values.mean()))
        if cvs:
            summary['architecture_consistency_cv_mean'] = float(np.mean(cvs))

    for structure in ('rv', 'myo', 'lv'):
        col_candidates = [f'ed_dice_{structure}', f'es_dice_{structure}', f'dice_{structure}']
        vals = []
        for col in col_candidates:
            if col in df.columns:
                vals.extend(pd.to_numeric(df[col], errors='coerce').dropna().tolist())
        if vals:
            summary[f'{structure}_dice_mean'] = float(np.mean(vals))

    return summary

def write_markdown_report(output_path: Path, run_name: str, model_selection: str, workflow_start: str, execution: Dict, gpu_profile: Dict, scientific: Dict):
    lines = []
    lines.append('# CASC Debug Report')
    lines.append('')
    lines.append('## Run Context')
    lines.append(f'- Run name: {run_name}')
    lines.append(f'- Models requested: {model_selection}')
    lines.append(f'- Workflow start: {workflow_start}')
    lines.append(f'- Workflow success: {execution.get("workflow_success")}')
    lines.append('')

    lines.append('## Execution & Reliability')
    lines.append(f'- Workflow duration (s): {execution.get("workflow_duration_seconds")}')
    lines.append(f'- Total tasks: {execution.get("total_tasks")}')
    lines.append(f'- Task success rate: {execution.get("task_success_rate"):.4f}' if not math.isnan(execution.get('task_success_rate', np.nan)) else '- Task success rate: NA')
    lines.append(f'- Retry rate: {execution.get("retry_rate"):.4f}' if not math.isnan(execution.get('retry_rate', np.nan)) else '- Retry rate: NA')
    lines.append(f'- Estimated CPU time (core-hours): {execution.get("cpu_time_hours_est"):.3f}' if not math.isnan(execution.get('cpu_time_hours_est', np.nan)) else '- Estimated CPU time (core-hours): NA')
    lines.append(f'- Estimated memory pressure (GB-hours from peak RSS): {execution.get("memory_gb_hours_est"):.3f}' if not math.isnan(execution.get('memory_gb_hours_est', np.nan)) else '- Estimated memory pressure (GB-hours from peak RSS): NA')
    lines.append(f'- Mean parallelism estimate: {execution.get("mean_parallelism_est"):.3f}' if not math.isnan(execution.get('mean_parallelism_est', np.nan)) else '- Mean parallelism estimate: NA')
    throughput = execution.get('patient_throughput_per_hour', np.nan)
    lines.append(f'- Patient throughput (patients/hour): {throughput:.3f}' if not math.isnan(throughput) else '- Patient throughput (patients/hour): NA')
    lines.append('')

    lines.append('## GPU Scalability')
    lines.append(f'- GPU task count: {gpu_profile.get("gpu_task_count")}')
    gpu_fraction = gpu_profile.get('gpu_task_fraction', np.nan)
    lines.append(f'- GPU task fraction: {gpu_fraction:.4f}' if not math.isnan(gpu_fraction) else '- GPU task fraction: NA')
    lines.append(f'- Estimated GPU walltime (hours): {gpu_profile.get("gpu_walltime_hours_est"):.3f}')
    avg_gpu = gpu_profile.get('avg_gpu_concurrency_est', np.nan)
    lines.append(f'- Average GPU concurrency estimate: {avg_gpu:.3f}' if not math.isnan(avg_gpu) else '- Average GPU concurrency estimate: NA')
    lines.append('')

    lines.append('## Scientific Utility')
    lines.append(f'- Input patient count: {scientific.get("n_patients_input")}')
    lines.append(f'- Evaluated patient count: {scientific.get("n_patients_with_metrics")}')
    coverage = scientific.get('coverage_rate', np.nan)
    lines.append(f'- Coverage rate: {coverage:.4f}' if not math.isnan(coverage) else '- Coverage rate: NA')
    lines.append(f'- Models evaluated: {scientific.get("n_models_evaluated")}')
    lines.append(f'- Best model by mean overall Dice: {scientific.get("best_model_overall_dice")} ({scientific.get("best_model_dice_value"):.4f})' if not math.isnan(scientific.get('best_model_dice_value', np.nan)) else '- Best model by mean overall Dice: NA')
    spread = scientific.get('cross_model_dice_spread', np.nan)
    lines.append(f'- Cross-model Dice spread: {spread:.4f}' if not math.isnan(spread) else '- Cross-model Dice spread: NA')
    consistency = scientific.get('architecture_consistency_cv_mean', np.nan)
    lines.append(f'- Architecture consistency CV (lower is better): {consistency:.4f}' if not math.isnan(consistency) else '- Architecture consistency CV (lower is better): NA')
    for structure in ('rv', 'myo', 'lv'):
        val = scientific.get(f'{structure}_dice_mean', np.nan)
        lines.append(f'- Mean {structure.upper()} Dice: {val:.4f}' if not math.isnan(val) else f'- Mean {structure.upper()} Dice: NA')

    output_path.write_text('\n'.join(lines) + '\n')

def make_figures(task_profile: pd.DataFrame, gpu_profile: Dict, metrics_df: pd.DataFrame, figures_dir: Path):
    figures_dir.mkdir(parents=True, exist_ok=True)

    if not task_profile.empty:
        top = task_profile.head(12)
        plt.figure(figsize=(12, 6))
        plt.bar(top['process'], top['runtime_hours_total'], color='#4e79a7')
        plt.title('Top Process Runtime Contribution')
        plt.ylabel('Runtime (hours)')
        plt.xlabel('Process')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(figures_dir / 'runtime_by_process.png', dpi=160)
        plt.close()

    breakdown = gpu_profile.get('gpu_process_breakdown', [])
    if breakdown:
        gpu_df = pd.DataFrame(breakdown)
        plt.figure(figsize=(10, 5))
        plt.bar(gpu_df['process'], gpu_df['gpu_walltime_hours'], color='#f28e2b')
        plt.title('Estimated GPU Walltime by Process')
        plt.ylabel('GPU walltime (hours)')
        plt.xlabel('Process')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(figures_dir / 'gpu_walltime_by_process.png', dpi=160)
        plt.close()

    if not metrics_df.empty and 'overall_dice_mean' in metrics_df.columns and 'model' in metrics_df.columns:
        model_perf = metrics_df.groupby('model', as_index=False)['overall_dice_mean'].mean().sort_values('overall_dice_mean', ascending=False)
        plt.figure(figsize=(10, 5))
        plt.bar(model_perf['model'], model_perf['overall_dice_mean'], color='#59a14f')
        plt.title('Mean Overall Dice by Model')
        plt.ylabel('Overall Dice')
        plt.xlabel('Model')
        plt.ylim(0, 1)
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.savefig(figures_dir / 'overall_dice_by_model.png', dpi=160)
        plt.close()

# Build CASC-like trace dataframe from notebook events
trace_df = pd.DataFrame(DEBUG_EVENTS) if len(DEBUG_EVENTS) else pd.DataFrame(columns=['name', 'status', 'realtime', '%cpu', 'peak_rss', 'label', 'attempt', 'submit'])
if not trace_df.empty:
    trace_df['cpus'] = 1

# Build metrics snapshot in CASC shape
metrics_snapshot = patient_df.copy() if 'patient_df' in globals() else pd.DataFrame()
if not metrics_snapshot.empty:
    metrics_snapshot['model'] = 'vsa3l__model'
    metrics_snapshot['architecture'] = 'vsa3l'

# Compute summary blocks
workflow_duration_seconds = derive_makespan_seconds(trace_df)
workflow_duration_str = f"{workflow_duration_seconds:.3f}s" if np.isfinite(workflow_duration_seconds) else ""
execution = compute_execution_metrics(trace_df, workflow_duration_str, str(True))
gpu_profile = build_gpu_profile(trace_df, execution.get('workflow_duration_seconds', np.nan))
n_patients_input = len(list(Path(DATA_DIR).glob('patient*')))
scientific = compute_scientific_metrics(metrics_snapshot, n_patients_input=n_patients_input)

# Notebook context: these Nextflow scheduler/cluster metrics are not directly meaningful
execution['retry_rate'] = np.nan
execution['cpu_time_hours_est'] = np.nan
execution['memory_gb_hours_est'] = np.nan
execution['mean_parallelism_est'] = np.nan
execution['patient_throughput_per_hour'] = np.nan
gpu_profile['avg_gpu_concurrency_est'] = np.nan

summary = {
    'run_name': RUN_NAME,
    'workflow_start': WORKFLOW_START,
    'models_requested': 'VSA3L',
    'execution': execution,
    'gpu': {k: v for k, v in gpu_profile.items() if k != 'gpu_process_breakdown'},
    'scientific': scientific,
}

# Persist CASC-style artifacts
task_profile = build_task_profile(trace_df)
gpu_breakdown = pd.DataFrame(gpu_profile.get('gpu_process_breakdown', []))

with open(DEBUG_TEXT_DIR / 'debug_metrics_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)

if not task_profile.empty:
    task_profile.to_csv(DEBUG_TEXT_DIR / 'task_profile.csv', index=False)
if not gpu_breakdown.empty:
    gpu_breakdown.to_csv(DEBUG_TEXT_DIR / 'gpu_profile.csv', index=False)
if not metrics_snapshot.empty:
    preferred_cols = [
        'patient_id', 'model', 'architecture',
        'ed_dice_rv', 'ed_hd95_rv', 'ed_dice_myo', 'ed_hd95_myo', 'ed_dice_lv', 'ed_hd95_lv',
        'es_dice_rv', 'es_hd95_rv', 'es_dice_myo', 'es_hd95_myo', 'es_dice_lv', 'es_hd95_lv',
        'overall_dice_mean'
    ]
    present_cols = [c for c in preferred_cols if c in metrics_snapshot.columns]
    metrics_snapshot[present_cols].to_csv(DEBUG_TEXT_DIR / 'metrics_snapshot.csv', index=False)

write_markdown_report(
    output_path=DEBUG_TEXT_DIR / 'debug_report.md',
    run_name=RUN_NAME,
    model_selection='VSA3L',
    workflow_start=WORKFLOW_START,
    execution=execution,
    gpu_profile=gpu_profile,
    scientific=scientific,
)

make_figures(
    task_profile=task_profile,
    gpu_profile=gpu_profile,
    metrics_df=metrics_snapshot,
    figures_dir=DEBUG_FIGURES_DIR,
)

print(f"Debug summary written to: {DEBUG_TEXT_DIR / 'debug_metrics_summary.json'}")
print(f"Debug markdown report written to: {DEBUG_TEXT_DIR / 'debug_report.md'}")
print(f"Debug text directory: {DEBUG_TEXT_DIR}")
print(f"Debug figures directory: {DEBUG_FIGURES_DIR}")